<a href="https://colab.research.google.com/github/V3GA5/CYB2423_Project/blob/main/Task2_Integrity_HMAC_RSA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import csv
import io
import json
import hashlib
import hmac
import secrets
import zipfile
from pathlib import Path
from google.colab import files
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.exceptions import InvalidSignature
from decimal import Decimal, InvalidOperation

uploaded = files.upload()
source_name = next((name for name in uploaded if name.lower().endswith(".csv")), None)
if source_name is None:
    raise ValueError("Select University_Scholarship_Applications_Dataset.csv")
records = list(csv.DictReader(io.StringIO(uploaded[source_name].decode("utf-8-sig"))))
expected_fields = ["ApplicationID", "StudentID", "DecisionStatus", "ApprovedAmountAED", "ReviewerID", "DecisionDate"]
# 1. Confirm the expected number of records
if len(records) != 96:
    raise ValueError(f"Expected 96 records, found {len(records)}")

# 2. Confirm that all required columns exist
missing_columns = [
    field for field in expected_fields
    if field not in records[0]
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# 3. Check that every application has an ApplicationID
if any(
    not str(record.get("ApplicationID") or "").strip()
    for record in records
):
    raise ValueError("Some records have a missing ApplicationID")

# 4. Confirm that ApplicationID values are unique
application_ids = [
    record["ApplicationID"].strip()
    for record in records
]

if len(set(application_ids)) != len(application_ids):
    raise ValueError("ApplicationID values must be unique")

# 5. Validate the decision status values
valid_statuses = {
    "APPROVED",
    "DECLINED",
    "PENDING",
    "UNDER_REVIEW"
}

invalid_statuses = {
    record["DecisionStatus"]
    for record in records
    if record["DecisionStatus"] not in valid_statuses
}

if invalid_statuses:
    raise ValueError(f"Invalid decision statuses: {invalid_statuses}")

print("DATASET VALIDATION")
print("Total records:", len(records))
print("Required columns: PASS")
print("ApplicationID completeness: PASS")
print("Unique ApplicationIDs: PASS")
print("Decision statuses: PASS")

def canonical_bytes(record):
    return json.dumps(record, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")

print("Dataset:", source_name)
print("Application records:", len(records))
print("Dataset fields:", len(records[0]))

Saving University_Scholarship_Applications_Dataset.csv to University_Scholarship_Applications_Dataset.csv
DATASET VALIDATION
Total records: 96
Required columns: PASS
ApplicationID completeness: PASS
Unique ApplicationIDs: PASS
Decision statuses: PASS
Dataset: University_Scholarship_Applications_Dataset.csv
Application records: 96
Dataset fields: 18


In [2]:
record_hashes = [(record["ApplicationID"], hashlib.sha256(canonical_bytes(record)).hexdigest()) for record in records]
with open("Task2_All_Record_SHA256.csv", "w", newline="", encoding="utf-8") as output:
    writer = csv.writer(output)
    writer.writerow(["ApplicationID", "SHA256"])
    writer.writerows(record_hashes)

print("SHA-256 RECORD HASHES")
print("ApplicationID | SHA256")
for application_id, digest in record_hashes[:10]:
    print(f"{application_id} | {digest}")
print("Total records hashed:", len(record_hashes))

SHA-256 RECORD HASHES
ApplicationID | SHA256
APP-2026-0065 | 5c66fe15fd813f2f06098d4e21ecfd2d3f501928115d1faafa822abcf7c7f588
APP-2026-0055 | a3cf06e77a91c0dd3ed25793b783c5ca3431ff50e11e1e2ae9cb138059b0293a
APP-2026-0082 | 688b890008bf202711802eb0f3fa9e3a1b9086717119697d2e957a849de71ddd
APP-2026-0051 | 9932fc45036e68daeb05d21cd19187a943394fcb1a1e18e19bbdc75afc25db5d
APP-2026-0009 | 48463ac46f93055bee8a7638c5b6ca248e8a46138543e74ace257d9db9da0887
APP-2026-0067 | 08f66eb6b137f0399e77044ed34aeea7bc92b1da8b8cf3179418c6641047b9d3
APP-2026-0040 | 6ade470c09a8e99e950b1390cd9e024e376689718a33d132519b8ece096e85fd
APP-2026-0047 | 7d6d11fdfc686df6fdec376d868c9cef3a6e338311fd75eca7f46787b66db5e8
APP-2026-0028 | a3627870c91f0891e15660d2d0d3d2e9a97f2523c1499f7574b60b6e51fc5add
APP-2026-0012 | 0b1b6517a28ec01c26ee0992d8e430e469f4358f416666df57b5504f6ba4e1c2
Total records hashed: 96


In [3]:
selected_application_id = "APP-2026-0065"
chosen = next((record for record in records if record["ApplicationID"] == selected_application_id), None)
if chosen is None:
    raise ValueError("Selected application was not found")

# Check required decision fields
required_decision_fields = [
    "ApplicationID",
    "StudentID",
    "DecisionStatus",
    "ApprovedAmountAED",
    "ReviewerID",
    "DecisionDate"
]

missing_values = [
    field for field in required_decision_fields
    if not str(chosen.get(field) or "").strip()
]

if missing_values:
    raise ValueError(
        f"Selected decision has missing fields: {missing_values}"
    )

# Confirm that the decision is finalized
if chosen["DecisionStatus"] not in {"APPROVED", "DECLINED"}:
    raise ValueError("The selected decision is not finalized")

# Validate the approved amount
try:
    approved_amount = Decimal(chosen["ApprovedAmountAED"])

    if not approved_amount.is_finite() or approved_amount < 0:
        raise ValueError("Approved amount must be a valid non-negative number")

    if chosen["DecisionStatus"] == "APPROVED" and approved_amount <= 0:
        raise ValueError("An approved decision must have a positive amount")

except (InvalidOperation, TypeError):
    raise ValueError("ApprovedAmountAED must contain a valid numeric value")

print("Selected decision validation: PASS")
print("Application:", chosen["ApplicationID"])
print("Decision status:", chosen["DecisionStatus"])
print("Approved amount:", approved_amount)

decision = {field: chosen[field] for field in expected_fields}
serialized_decision = canonical_bytes(decision)
decision_digest = hashlib.sha256(serialized_decision).hexdigest()
Path("Task2_Selected_Decision.json").write_text(json.dumps(decision, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print("FINALIZED DECISION")
print(json.dumps(decision, indent=2, ensure_ascii=False))
print("Canonical decision SHA-256:", decision_digest)

Selected decision validation: PASS
Application: APP-2026-0065
Decision status: APPROVED
Approved amount: 12000
FINALIZED DECISION
{
  "ApplicationID": "APP-2026-0065",
  "StudentID": "STU-00065",
  "DecisionStatus": "APPROVED",
  "ApprovedAmountAED": "12000",
  "ReviewerID": "OFF-02",
  "DecisionDate": "15/10/2026"
}
Canonical decision SHA-256: 4cbd7c0be9860a17779d8502f303ecdff3e020e98d2589aab7199e40f34e207d


In [4]:
shared_secret = secrets.token_bytes(32)
original_mac = hmac.new(shared_secret, serialized_decision, hashlib.sha256).digest()
private_key = rsa.generate_private_key(public_exponent=65537, key_size=3072)
public_key = private_key.public_key()
original_signature = private_key.sign(serialized_decision, padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH), hashes.SHA256())

def verify_mac(candidate):
    candidate_mac = hmac.new(shared_secret, canonical_bytes(candidate), hashlib.sha256).digest()
    return hmac.compare_digest(original_mac, candidate_mac)

def verify_signature(candidate):
    try:
        public_key.verify(original_signature, canonical_bytes(candidate), padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH), hashes.SHA256())
        return True
    except InvalidSignature:
        return False

tampered_amount = dict(decision)
tampered_amount["ApprovedAmountAED"] = str(Decimal(decision["ApprovedAmountAED"]) + Decimal("1000"))
tampered_status = dict(decision)
tampered_status["DecisionStatus"] = "DECLINED" if decision["DecisionStatus"] == "APPROVED" else "APPROVED"
cases = [("Original decision", decision), ("Amount changed", tampered_amount), ("Status changed", tampered_status)]
test_results = [(label, "PASS" if verify_mac(candidate) else "FAIL", "PASS" if verify_signature(candidate) else "FAIL") for label, candidate in cases]
assert test_results == [("Original decision", "PASS", "PASS"), ("Amount changed", "FAIL", "FAIL"), ("Status changed", "FAIL", "FAIL")]

print("HMAC-SHA-256:", original_mac.hex())
print("RSA signature bytes:", len(original_signature))
print("VERIFICATION RESULTS")
print("Case | HMAC | RSA-PSS")
for label, mac_result, rsa_result in test_results:
    print(f"{label} | {mac_result} | {rsa_result}")
print("Original approved amount (AED):", decision["ApprovedAmountAED"])
print("Modified approved amount (AED):", tampered_amount["ApprovedAmountAED"])
print("Modified decision status:", tampered_status["DecisionStatus"])

HMAC-SHA-256: c454bdb9bc46a765876f008f5bae4b71ac212c7d8f51982f7c015cad23f59c7d
RSA signature bytes: 384
VERIFICATION RESULTS
Case | HMAC | RSA-PSS
Original decision | PASS | PASS
Amount changed | FAIL | FAIL
Status changed | FAIL | FAIL
Original approved amount (AED): 12000
Modified approved amount (AED): 13000
Modified decision status: DECLINED


In [5]:
with open("Task2_Integrity_Verification.csv", "w", newline="", encoding="utf-8") as output:
    writer = csv.writer(output)
    writer.writerow(["TestCase", "HMAC_SHA256", "RSA_PSS_SHA256"])
    writer.writerows(test_results)
Path("Task2_Decision_HMAC_SHA256.txt").write_text(original_mac.hex() + "\n", encoding="utf-8")
Path("Task2_Decision_Signature.bin").write_bytes(original_signature)
Path("Task2_RSA_Public_Key.pem").write_bytes(public_key.public_bytes(encoding=serialization.Encoding.PEM, format=serialization.PublicFormat.SubjectPublicKeyInfo))
summary = ["Task 2: Integrity, Message Authentication and Digital Signatures", f"Source: {source_name}", f"Records hashed: {len(record_hashes)}", f"Selected application: {selected_application_id}", f"Decision SHA-256: {decision_digest}", "", "Case | HMAC-SHA-256 | RSA-PSS-SHA-256"]
summary.extend(f"{name} | {mac_result} | {rsa_result}" for name, mac_result, rsa_result in test_results)
Path("Task2_Verification_Summary.txt").write_text("\n".join(summary) + "\n", encoding="utf-8")
output_names = ["Task2_All_Record_SHA256.csv", "Task2_Selected_Decision.json", "Task2_Integrity_Verification.csv", "Task2_Decision_HMAC_SHA256.txt", "Task2_Decision_Signature.bin", "Task2_RSA_Public_Key.pem", "Task2_Verification_Summary.txt"]
with zipfile.ZipFile("Task2_Evidence_Files.zip", "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in output_names:
        archive.write(name)
print("Created:", ", ".join(output_names))
print("Evidence archive: Task2_Evidence_Files.zip")
files.download("Task2_Evidence_Files.zip")

Created: Task2_All_Record_SHA256.csv, Task2_Selected_Decision.json, Task2_Integrity_Verification.csv, Task2_Decision_HMAC_SHA256.txt, Task2_Decision_Signature.bin, Task2_RSA_Public_Key.pem, Task2_Verification_Summary.txt
Evidence archive: Task2_Evidence_Files.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>